In [1]:
"""
transformer_code.py
--------------------
Minimal, well-commented PyTorch implementation of the core building
blocks of a Transformer (Vaswani et al., 2017 - "Attention Is All You Need").

This is meant as a companion to the notes document, not a production model.
It implements, from scratch:
    1. Scaled Dot-Product Attention
    2. Multi-Head Attention
    3. Positional Encoding (sinusoidal)
    4. Position-wise Feed-Forward Network
    5. A single Transformer Encoder Block (pre-norm style)
    6. A tiny stacked Encoder + a toy run-through

Requires: pip install torch --break-system-packages
"""



'\ntransformer_code.py\n--------------------\nMinimal, well-commented PyTorch implementation of the core building\nblocks of a Transformer (Vaswani et al., 2017 - "Attention Is All You Need").\n\nThis is meant as a companion to the notes document, not a production model.\nIt implements, from scratch:\n    1. Scaled Dot-Product Attention\n    2. Multi-Head Attention\n    3. Positional Encoding (sinusoidal)\n    4. Position-wise Feed-Forward Network\n    5. A single Transformer Encoder Block (pre-norm style)\n    6. A tiny stacked Encoder + a toy run-through\n\nRequires: pip install torch --break-system-packages\n'

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
# ---------------------------------------------------------------------------
# 1. Scaled Dot-Product Attention
# ---------------------------------------------------------------------------
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q, k, v: (batch, heads, seq_len, d_k)
    mask:    (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len) - optional

    Returns: (output, attention_weights)
    """
    d_k = q.size(-1)

    # 1. Similarity between every query and every key
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    # 2. Mask out positions we shouldn't attend to (e.g. future tokens, padding)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # 3. Turn scores into a probability distribution over keys
    attn_weights = F.softmax(scores, dim=-1)

    # 4. Weighted sum of values = the "context" the model pulls in
    output = torch.matmul(attn_weights, v)
    return output, attn_weights


In [4]:
# ---------------------------------------------------------------------------
# 2. Multi-Head Attention
# ---------------------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    """
    Instead of computing one attention function with d_model-dim keys/values,
    we project into `num_heads` smaller subspaces, attend in parallel, then
    concatenate. This lets different heads specialize (syntax, position,
    coreference, etc.).
    """

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)  # output projection

    def split_heads(self, x, batch_size):
        # (batch, seq_len, d_model) -> (batch, heads, seq_len, d_k)
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        q = self.split_heads(self.w_q(q), batch_size)
        k = self.split_heads(self.w_k(k), batch_size)
        v = self.split_heads(self.w_v(v), batch_size)

        attn_output, attn_weights = scaled_dot_product_attention(q, k, v, mask)

        # Merge heads back: (batch, heads, seq_len, d_k) -> (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )

        return self.w_o(attn_output), attn_weights


In [5]:
# ---------------------------------------------------------------------------
# 3. Positional Encoding (sinusoidal - the original, parameter-free version)
# ---------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    """
    Self-attention has no built-in notion of order (it's permutation
    invariant), so we inject position information by adding a fixed
    sinusoidal pattern to the input embeddings.
    """

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, : x.size(1)]

